# Unit 11: Melakukan Proses Review Pemodelan
**Kode unit:** J.62DMI00.015.1 · SKKNI No. 299 Tahun 2020 · Proyek: *Prediksi Arah Harian IHSG*

**Tujuan unit ini:** meninjau ulang seluruh proses, menilai hasil secara jujur, dan menentukan langkah berikutnya.

In [1]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110

In [2]:
with open('../models/hasil_evaluasi.json') as f:
    hasil = json.load(f)
tabel = pd.DataFrame(hasil['hasil'])
target = pd.DataFrame(hasil['target'])
print('Model akhir:', hasil['model'], '| data uji:', hasil['jumlah_hari_uji'], 'hari')
display(tabel.round(3))
display(target)

Model akhir: Logistic Regression | data uji: 812 hari


,Model,Accuracy,F1-macro,ROC-AUC
0,Iterasi 2: Logistic Regression (5 fitur),0.558,0.554,0.583
1,Selalu Naik (baseline),0.522,0.343,0.500
2,Ikuti S&P 500 (baseline),0.546,0.543,0.575
3,Iterasi 1: Random Forest (6 fitur),0.543,0.535,0.577


,Target,Nilai,Tercapai
0,"Accuracy ≥ 0,55",0.558,Ya
1,"F1-macro ≥ 0,55",0.554,Ya
2,"ROC-AUC ≥ 0,58",0.583,Ya
3,"Accuracy > baseline ""Selalu Naik""",0.036,Ya


## 1. Review proses
| Tahap | Yang dilakukan | Catatan |
|---|---|---|
| Bisnis & teknis (Unit 1–2) | Pertanyaan bisnis dan target ukuran ditetapkan di awal | Target dibuat sebelum melihat hasil |
| Data (Unit 3–7) | Telaah, validasi, pembersihan, pembuatan fitur | 321 baris/nilai bermasalah ditemukan dan diperbaiki |
| Model (Unit 8–9) | 3 model dan 2 baseline dibandingkan, dalam 2 iterasi fitur | Validasi memakai urutan waktu |
| Evaluasi (Unit 10) | Data uji dipakai pada 2 iterasi | Hasil kedua iterasi dilaporkan apa adanya |

## 2. Dua iterasi
| Iterasi | Fitur | Model terpilih (validasi silang) | Hasil pada data uji | Target tercapai |
|---|---|---|---|---|
| 1 | 6, dengan kurs USD/IDR | Random Forest (0,595) | accuracy 0,543 · F1 0,535 · ROC-AUC 0,577 | 1 dari 4 |
| 2 | 5, tanpa kurs | Logistic Regression (0,599) | accuracy 0,558 · F1 0,554 · ROC-AUC 0,583 | 4 dari 4 |

Hasil iterasi 1 ditinjau: model hanya sedikit lebih baik daripada tebakan "selalu naik" dan tidak lebih baik daripada
aturan "ikuti S&P 500". Alasan kurs dikeluarkan berasal dari data latih: korelasinya dengan IHSG besok hampir nol
(−0,02, Unit 3), pengaruhnya paling kecil di model iterasi 1 (Random Forest, Unit 9), dan validasi silang tidak turun tanpa kurs
(0,595 → 0,599). Namun keputusan ini diambil **setelah** hasil uji iterasi 1 terlihat, sehingga hasil iterasi 2 bisa
sedikit terlalu optimistis.

## 3. Kelebihan
- Memakai **data nyata** 16 tahun dan alur kerja yang lengkap.
- Masalah kualitas data ditemukan dan diperbaiki secara terdokumentasi.
- Pengujian jujur: data dibagi menurut waktu dan model dibandingkan dengan tebakan sederhana.
- Model sederhana dan mudah dijelaskan.

## 4. Kekurangan
- Arah pasar saham harian sangat sulit ditebak, sehingga kinerja model terbatas.
- Kekuatan model sebagian besar berasal dari **S&P 500 semalam**; aturan sederhana "ikuti S&P 500" hampir sama baiknya.
- Data uji dipakai pada dua iterasi, sehingga hasil akhir perlu dikonfirmasi dengan data baru.
- Model belum memperhitungkan biaya transaksi, jadi **bukan strategi trading**.

In [3]:
semua_tercapai = (target['Tercapai'] == 'Ya').all()
if semua_tercapai:
    keputusan = 'LANJUT ke uji coba terbatas sebagai alat bantu analisis (bukan sinyal trading).'
else:
    belum = ', '.join(target.loc[target['Tercapai'] == 'Tidak', 'Target'])
    keputusan = f'BELUM LANJUT. Target yang belum tercapai: {belum}. Perbaiki model dulu (lihat rekomendasi).'
print('KEPUTUSAN:', keputusan)

KEPUTUSAN: LANJUT ke uji coba terbatas sebagai alat bantu analisis (bukan sinyal trading).


**Catatan hasil:** pada iterasi 2 semua target tercapai, tetapi selisihnya **tipis** (ROC-AUC 0,583 vs target 0,58).
Langkah berikutnya adalah **uji coba paralel** dengan data baru setelah September 2026, bukan langsung dipakai.

Periode uji (Apr 2023 – Sep 2026) juga mencakup gejolak besar: IHSG turun **41,5%** dari puncaknya 9.134,7
(20 Jan 2026) ke 5.342,1 (8 Jun 2026).

## 5. Rekomendasi perbaikan
1. Tambah data **bursa Asia** yang buka sebelum BEI (Nikkei, Hang Seng) sebagai fitur.
2. Latih ulang model **setiap bulan** agar mengikuti kondisi pasar terbaru.
3. Uji keuntungan nyata dengan memperhitungkan harga pembukaan dan biaya transaksi.
4. Perbaikan berikutnya diuji dengan **data baru** (setelah September 2026), bukan data uji yang sudah dipakai.

## 6. Rencana penerapan
| Tahap | Waktu | Kegiatan |
|---|---|---|
| Uji coba | Bulan 1–2 | Prediksi dibuat otomatis tiap pagi, dicatat tanpa dipakai |
| Evaluasi | Bulan 3 | Bandingkan prediksi dengan hasil nyata |
| Penggunaan | Bulan 4–6 | Dipakai sebagai bahan laporan pagi bila hasil memadai |
| Pemantauan | Rutin | Accuracy dihitung ulang setiap bulan; model dilatih ulang bila menurun |

---
*Proyek ini adalah portofolio kompetensi data science, bukan rekomendasi investasi.*